In [1]:
import pandas as pd
import numpy as np

import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

stemmer = PorterStemmer()

In [2]:
#  Load data :
df = pd.read_csv("emails.csv")
df.head(3)

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1


In [3]:
# replace :
df['text'] = df['text'].str.replace("Subject:"," ",regex=False)

In [4]:
df.head()

,text,spam
0,naturally irresistible your corporate identi...,1
1,the stock trading gunslinger fanny is merri...,1
2,unbelievable new homes made easy im wanting...,1
3,4 color printing special request additional...,1
4,"do not have money , get software cds from he...",1


In [5]:
# clean data

import re


def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^ a-zA-Z]'," ", text)
  text = text.split()

  return text


df['clean_text'] = df['text'].apply(clean_text)

In [6]:
df.head(3)

,text,spam,clean_text
0,naturally irresistible your corporate identi...,1,"[naturally, irresistible, your, corporate, ide..."
1,the stock trading gunslinger fanny is merri...,1,"[the, stock, trading, gunslinger, fanny, is, m..."
2,unbelievable new homes made easy im wanting...,1,"[unbelievable, new, homes, made, easy, im, wan..."


In [7]:
# perform stop word
nltk.download('stopwords')
stop = set(stopwords.words("english"))

df['token'] = df['clean_text'].apply(
    lambda words :[word for word in words if word not in stop]
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gamin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
# Root Finding :

df['token'] = df['token'].apply(
    lambda words:[stemmer.stem(word)  for word in words ]
)

In [12]:
# vectorization :
df['final_text'] = df['token'].apply( lambda x : " ".join(x))
X=tfidf.fit_transform(df['final_text'])
y = df['spam']

In [13]:
df1 = pd.DataFrame(X.toarray(), columns=tfidf.get_feature_names_out())

df1.head()

,aa,aaa,aaaenerfax,aadedeji,aagraw,aal,aaldou,aaliyah,aall,aanalysi,...,zwzm,zxghlajf,zyban,zyc,zygoma,zymg,zzmacmac,zzn,zzncacst,zzzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# model build :
# logistics  regression
# multimonialNB

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)


model = MultinomialNB()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(model.score(X_test,y_test))
print(model.score(X_train,y_train))

0.8821989528795812
0.919249236141423


In [16]:
# evalutation time :

from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.86      1.00      0.93       856
           1       1.00      0.53      0.70       290

    accuracy                           0.88      1146
   macro avg       0.93      0.77      0.81      1146
weighted avg       0.90      0.88      0.87      1146

[[856   0]
 [135 155]]
0.8821989528795812


In [17]:
df['spam'].value_counts(normalize=True)*100

spam
0    76.117318
1    23.882682
Name: proportion, dtype: float64

In [18]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced')

lr.fit(X_train,y_train)

lr_pred = lr.predict(X_test)
print(classification_report(y_test,lr_pred))
print(confusion_matrix(y_test,lr_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       856
           1       0.97      1.00      0.98       290

    accuracy                           0.99      1146
   macro avg       0.98      0.99      0.99      1146
weighted avg       0.99      0.99      0.99      1146

[[847   9]
 [  1 289]]


In [19]:
# save model
import pickle

pickle.dump(lr,open("model.pkl","wb"))
pickle.dump(tfidf,open("vector.pkl","wb"))